In [6]:
"""
Phase 1: Semantic Embedding & Clustering
==========================================
AD-BoN routing framework — maps prompt semantics to clusters so the
runtime router can reason about prompt "regions" (topic/complexity
neighborhoods) rather than raw text.

INPUT:  dataset_clean.csv   (columns: ID, Prompt, Task Type, Difficulty)
OUTPUT: dataset_clustered.csv (adds a cluster_id column)
        kmeans_model.joblib   (trained KMeans model)
        cluster_centroids.npy (raw centroid coordinates)
        k_selection_curve.png (Elbow + Silhouette plot)
        cluster_visualization.png (t-SNE scatter, colored by cluster,
                                    styled by Difficulty)

HOW TO USE
----------
1. Edit CONFIG below (input/output paths, K range, model name).
2. pip install pandas scikit-learn sentence-transformers tqdm matplotlib seaborn joblib torch
3. python phase1_clustering.py
"""

from __future__ import annotations

import sys
import logging
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("phase1")

# ============================== CONFIG ==============================
INPUT_PATH = "dataset_clean.csv"
OUTPUT_CSV_PATH = "dataset_clustered.csv"

KMEANS_MODEL_PATH = "kmeans_model.joblib"
CENTROIDS_PATH = "cluster_centroids.npy"
K_CURVE_PLOT_PATH = "k_selection_curve.png"
CLUSTER_PLOT_PATH = "cluster_visualization.png"

PROMPT_COL = "Prompt"
DIFFICULTY_COL = "Difficulty"

EMBEDDING_MODEL_NAME = "all-MiniLM-L12-v2"
EMBEDDING_BATCH_SIZE = 64

K_RANGE = range(3, 11)          # K in [3, 10] inclusive
RANDOM_STATE = 42               # fixed for reproducibility (KMeans + t-SNE)
TSNE_PERPLEXITY = 30
# ======================================================================


def load_dataset(path: str) -> pd.DataFrame:
    """Load the cleaned dataset, with clear errors on missing file/columns."""
    p = Path(path)
    if not p.exists():
        log.error(f"Input file not found: {p.resolve()}")
        sys.exit(1)

    df = pd.read_csv(p)

    required_cols = {PROMPT_COL, DIFFICULTY_COL}
    missing = required_cols - set(df.columns)
    if missing:
        log.error(f"Missing required column(s): {missing}. Found columns: {df.columns.tolist()}")
        sys.exit(1)

    if df[PROMPT_COL].isna().any():
        n_missing = df[PROMPT_COL].isna().sum()
        log.warning(f"{n_missing} rows have empty '{PROMPT_COL}' values — dropping them.")
        df = df.dropna(subset=[PROMPT_COL]).reset_index(drop=True)

    log.info(f"Loaded {len(df)} rows from {p.name}.")
    return df


def generate_embeddings(prompts: list[str]) -> np.ndarray:
    """
    Encode prompts into dense semantic vectors using sentence-transformers.
    Falls back to CPU automatically if no CUDA GPU is available.
    """
    try:
        import torch
        from sentence_transformers import SentenceTransformer
    except ImportError as e:
        log.error(
            "Missing dependency. Install with: "
            "pip install sentence-transformers torch"
        )
        raise e

    device = "cuda" if torch.cuda.is_available() else "cpu"
    log.info(f"Loading embedding model '{EMBEDDING_MODEL_NAME}' on device: {device}")

    try:
        model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)
    except Exception as e:
        log.error(f"Failed to load embedding model: {e}")
        raise

    try:
        embeddings = model.encode(
            prompts,
            batch_size=EMBEDDING_BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
        )
    except RuntimeError as e:
        # Typically an out-of-memory error on GPU
        if "out of memory" in str(e).lower() and device == "cuda":
            log.warning("CUDA OOM — retrying on CPU with a smaller batch size.")
            import torch as _torch
            _torch.cuda.empty_cache()
            model = SentenceTransformer(EMBEDDING_MODEL_NAME, device="cpu")
            embeddings = model.encode(
                prompts,
                batch_size=max(8, EMBEDDING_BATCH_SIZE // 4),
                show_progress_bar=True,
                convert_to_numpy=True,
            )
        else:
            raise

    log.info(f"Generated embeddings with shape: {embeddings.shape}")
    return embeddings


def find_optimal_k(embeddings: np.ndarray, k_range: range) -> Tuple[int, dict]:
    """
    Evaluate KMeans across k_range using both inertia (Elbow) and
    Silhouette Score. Returns the K with the best Silhouette score
    (more reliable than Elbow's visual 'knee' for small N), plus
    the raw metrics for plotting.
    """
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score

    inertias = []
    silhouette_scores = []
    k_values = list(k_range)

    log.info(f"Evaluating K in {k_values} using Elbow + Silhouette methods...")

    for k in k_values:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init="auto")
        labels = km.fit_predict(embeddings)
        inertias.append(km.inertia_)
        score = silhouette_score(embeddings, labels)
        silhouette_scores.append(score)
        log.info(f"  K={k:2d} | inertia={km.inertia_:10.2f} | silhouette={score:.4f}")

    best_idx = int(np.argmax(silhouette_scores))
    best_k = k_values[best_idx]
    log.info(f"Selected K={best_k} (highest silhouette score: {silhouette_scores[best_idx]:.4f})")

    metrics = {
        "k_values": k_values,
        "inertias": inertias,
        "silhouette_scores": silhouette_scores,
    }
    return best_k, metrics


def plot_k_selection_curve(metrics: dict, save_path: str) -> None:
    """Plot Elbow (inertia) and Silhouette score curves side by side."""
    import matplotlib
    matplotlib.use("Agg")  # non-interactive backend, safe for headless runs
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(metrics["k_values"], metrics["inertias"], marker="o")
    axes[0].set_title("Elbow Method (Inertia)")
    axes[0].set_xlabel("K")
    axes[0].set_ylabel("Inertia")
    axes[0].grid(alpha=0.3)

    axes[1].plot(metrics["k_values"], metrics["silhouette_scores"], marker="o", color="darkorange")
    axes[1].set_title("Silhouette Score")
    axes[1].set_xlabel("K")
    axes[1].set_ylabel("Silhouette Score")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    try:
        plt.savefig(save_path, dpi=150)
        log.info(f"Saved K-selection curve to {save_path}")
    except OSError as e:
        log.error(f"Failed to save plot to {save_path}: {e}")
    finally:
        plt.close(fig)


def run_kmeans(embeddings: np.ndarray, k: int):
    """Fit final KMeans model with the chosen K and return model + labels."""
    from sklearn.cluster import KMeans

    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init="auto")
    labels = km.fit_predict(embeddings)
    log.info(f"Fitted final KMeans with K={k}. Cluster sizes: {np.bincount(labels).tolist()}")
    return km, labels


def save_model_artifacts(km, embeddings: np.ndarray) -> None:
    """Persist the trained KMeans model and raw centroid coordinates to disk."""
    import joblib

    try:
        joblib.dump(km, KMEANS_MODEL_PATH)
        log.info(f"Saved KMeans model to {KMEANS_MODEL_PATH}")
    except OSError as e:
        log.error(f"Failed to save KMeans model: {e}")
        raise

    try:
        np.save(CENTROIDS_PATH, km.cluster_centers_)
        log.info(f"Saved cluster centroids to {CENTROIDS_PATH}")
    except OSError as e:
        log.error(f"Failed to save centroids: {e}")
        raise


def visualize_clusters(
    embeddings: np.ndarray,
    cluster_labels: np.ndarray,
    difficulty_labels: pd.Series,
    save_path: str,
) -> None:
    """
    Project embeddings to 2D via t-SNE and plot, colored by cluster_id
    and styled by Difficulty, to visually check whether clusters track
    semantic/difficulty boundaries.
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.manifold import TSNE

    n_samples = embeddings.shape[0]
    perplexity = min(TSNE_PERPLEXITY, max(5, n_samples // 3))  # guard for small N

    log.info(f"Running t-SNE (perplexity={perplexity}) on {n_samples} points...")
    tsne = TSNE(
        n_components=2,
        random_state=RANDOM_STATE,
        perplexity=perplexity,
        init="pca",
    )
    coords_2d = tsne.fit_transform(embeddings)

    plot_df = pd.DataFrame({
        "x": coords_2d[:, 0],
        "y": coords_2d[:, 1],
        "cluster_id": cluster_labels.astype(str),
        "Difficulty": difficulty_labels.values,
    })

    plt.figure(figsize=(10, 8))
    sns.scatterplot(
        data=plot_df,
        x="x", y="y",
        hue="cluster_id",
        style="Difficulty",
        palette="tab10",
        s=70,
        alpha=0.8,
    )
    plt.title("Prompt Clusters (t-SNE projection)\nColor = cluster_id, Marker = Difficulty")
    plt.xlabel("t-SNE dim 1")
    plt.ylabel("t-SNE dim 2")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0.)
    plt.tight_layout()

    try:
        plt.savefig(save_path, dpi=150)
        log.info(f"Saved cluster visualization to {save_path}")
    except OSError as e:
        log.error(f"Failed to save plot to {save_path}: {e}")
    finally:
        plt.close()


def main() -> None:
    df = load_dataset(INPUT_PATH)
    prompts = df[PROMPT_COL].astype(str).tolist()

    embeddings = generate_embeddings(prompts)

    best_k, metrics = find_optimal_k(embeddings, K_RANGE)
    plot_k_selection_curve(metrics, K_CURVE_PLOT_PATH)

    km, cluster_labels = run_kmeans(embeddings, best_k)
    save_model_artifacts(km, embeddings)

    if DIFFICULTY_COL in df.columns:
        visualize_clusters(embeddings, cluster_labels, df[DIFFICULTY_COL], CLUSTER_PLOT_PATH)
    else:
        log.warning(f"'{DIFFICULTY_COL}' column not found — skipping styled visualization.")

    df["cluster_id"] = cluster_labels
    try:
        df.to_csv(OUTPUT_CSV_PATH, index=False)
        log.info(f"Saved cluster-labeled dataset to {OUTPUT_CSV_PATH}")
    except OSError as e:
        log.error(f"Failed to write output CSV: {e}")
        sys.exit(1)

    log.info("Phase 1 complete.")
    log.info(f"Cluster distribution:\n{df['cluster_id'].value_counts().sort_index()}")


if __name__ == "__main__":
    main()

2026-09-05 02:44:14,397 [INFO] Loaded 1162 rows from dataset_clean.csv.
2026-09-05 02:44:33,568 [INFO] Loading embedding model 'all-MiniLM-L12-v2' on device: cpu
2026-09-05 02:44:34,596 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L12-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-09-05 02:44:34,602 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-09-05 02:44:34,625 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L12-v2/a50ef00143b4d5391434df20ae11632588ac25be/modules.json "HTTP/1.1 200 OK"
2026-09-05 02:44:34,701 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L12-v2/a50ef00143b4d5391434df20ae11632588ac25be/modules.json "HTTP/1.1 200 OK"


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\abdul\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\abdul\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
2026-09-05 02:44:35,013 [INFO] HTTP Request: HEAD https://huggingface.co/s

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

2026-09-05 02:44:35,330 [INFO] Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L12-v2.
2026-09-05 02:44:35,555 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L12-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-09-05 02:44:35,570 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L12-v2/a50ef00143b4d5391434df20ae11632588ac25be/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-09-05 02:44:35,795 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L12-v2/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-09-05 02:44:35,814 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L12-v2/a50ef00143b4d5391434df20ae11632588ac25be/README.md "HTTP/1.1 200 OK"
2026-09-05 02:44:35,835 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cac

...

2026-09-05 02:44:52,957 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L12-v2/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-09-05 02:44:53,272 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L12-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-05 02:44:53,296 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L12-v2/a50ef00143b4d5391434df20ae11632588ac25be/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
2026-09-05 02:44:53,358 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L12-v2/a50ef00143b4d5391434df20ae11632588ac25be/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-09-05 02:44:53,624 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L12-v2/resolve/main/2_Normalize/config.json "HTTP/1.1 404 Not Found"
2026-09-05 02:44:53,872 [INFO] HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L12-v2 "HTTP/1.1 200 OK"


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

2026-09-05 02:44:59,825 [INFO] Generated embeddings with shape: (1162, 384)
2026-09-05 02:45:01,066 [INFO] Evaluating K in [3, 4, 5, 6, 7, 8, 9, 10] using Elbow + Silhouette methods...
2026-09-05 02:45:03,305 [INFO]   K= 3 | inertia=    926.19 | silhouette=0.0639
2026-09-05 02:45:03,389 [INFO]   K= 4 | inertia=    900.08 | silhouette=0.0666
2026-09-05 02:45:03,471 [INFO]   K= 5 | inertia=    873.59 | silhouette=0.0773
2026-09-05 02:45:03,560 [INFO]   K= 6 | inertia=    858.05 | silhouette=0.0791
2026-09-05 02:45:03,652 [INFO]   K= 7 | inertia=    841.51 | silhouette=0.0844
2026-09-05 02:45:03,745 [INFO]   K= 8 | inertia=    803.71 | silhouette=0.0906
2026-09-05 02:45:03,843 [INFO]   K= 9 | inertia=    804.06 | silhouette=0.0898
2026-09-05 02:45:03,951 [INFO]   K=10 | inertia=    777.80 | silhouette=0.1023
2026-09-05 02:45:03,953 [INFO] Selected K=10 (highest silhouette score: 0.1023)
2026-09-05 02:45:05,369 [INFO] Saved K-selection curve to k_selection_curve.png
2026-09-05 02:45:05,433

In [8]:
!pip install pandas numpy tqdm

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip
